# EXP-003 조건 목록 dev-small 확인

목적: 질의 조건 목록을 RRF 세 번째 목록으로 더한 hybrid(EXP-003)가 동작하는지, 조건이 없는 질의의 순위가 EXP-001과 같은지 dev-small에서 본다. 판정은 dev-full·multi-doc으로만 한다(`docs/experiments/EXP-003.md`). 질의 원문은 출력하지 않는다(D-10).

In [1]:
import json
from pathlib import Path

import pandas as pd

from rag.conditions import ConditionIndex
from rag.eval.metrics import gold_ranks, load_cfg, load_gold, min_docs, per_query, read_run
from rag.index import load_cfg as load_hydra

In [2]:
ROOT = Path.cwd().parent
LAYER = "dev-small"
RUNS = {
    "exp001": ROOT / "data/runs/kure-v1-fixed-512-64-hybrid-dev-small.csv",
    "exp003": ROOT / "data/runs/kure-v1-fixed-512-64-hybrid-cond-dev-small.csv",
}
CONDITIONS = ROOT / "data/conditions/summary_q.jsonl"

## 순위 파일

`uv run python -m rag.search +exp=exp003`로 만들었다. 질의마다 서로 다른 문서 10개가 있는지 본다.

In [3]:
runs = {name: read_run(path) for name, path in RUNS.items()}
for name, run in runs.items():
    per_q = run.groupby("qid")["doc_id"].nunique()
    print(name, run.shape, "질의", per_q.size, "질의당 문서 수", per_q.min(), "~", per_q.max())

exp001 (10390, 3) 질의 1039 질의당 문서 수 10 ~ 10
exp003 (10390, 3) 질의 1039 질의당 문서 수 10 ~ 10


## 조건이 없는 질의는 순위가 같은가

조건 목록이 비면 RRF 입력이 EXP-001과 같아 순위가 같아야 한다.

In [4]:
with CONDITIONS.open(encoding="utf-8") as f:
    conds = {r["qid"]: r["conditions"] for r in map(json.loads, f)}
qids = runs["exp001"]["qid"].unique()
has_cond = pd.Series({q: bool(conds[q]) for q in qids})
print("dev-small 질의", len(qids), "조건 있음", has_cond.sum())

dev-small 질의 1039 조건 있음 58


In [5]:
order = {name: run.sort_values(["qid", "rank"]).groupby("qid")["doc_id"].apply(tuple)
         for name, run in runs.items()}
same = order["exp001"] == order["exp003"]
pd.crosstab(has_cond, same, rownames=["조건 있음"], colnames=["순위 같음"])

순위 같음,False,True
조건 있음,,
False,0,981
True,31,27


## 지표 비교

In [6]:
cfg = load_cfg()
ev = cfg.eval
gold = load_gold(cfg, LAYER)
n_min = min_docs(ev.recall_ks, ev.mrr_k)
ranks = {name: gold_ranks(run, gold, n_min) for name, run in runs.items()}
scores = {name: per_query(r, ev.recall_ks, ev.mrr_k) for name, r in ranks.items()}
pd.DataFrame({name: s.mean() for name, s in scores.items()}).round(4)

,exp001,exp003
recall@1,0.9105,0.8989
recall@5,0.9808,0.9750
recall@10,0.9875,0.9865
mrr@10,0.9438,0.9353


조건이 있는 질의만 본다.

In [7]:
cond_q = has_cond[has_cond].index
pd.DataFrame({name: s.loc[cond_q].mean() for name, s in scores.items()}).round(4)

,exp001,exp003
recall@1,0.9483,0.7414
recall@5,1.0000,0.8966
recall@10,1.0000,0.9828
mrr@10,0.9741,0.8228


In [8]:
hit = pd.DataFrame({name: s["recall@5"] for name, s in scores.items()})
pd.crosstab(hit.loc[cond_q, "exp001"], hit.loc[cond_q, "exp003"],
            rownames=["exp001@5"], colnames=["exp003@5"])

exp003@5,0.0,1.0
exp001@5,,
1.0,6,52


## 이탈 질의의 조건

EXP-001에서 5위 안에 있던 정답이 밀려난 질의마다 뽑힌 조건 필드와, 정답 문서가 그 조건에 맞는지 본다. 정답 문서가 맞은 필드가 없으면 틀리게 뽑힌 조건이 다른 문서를 끌어올린 경우다.

In [9]:
hcfg = load_hydra(["+exp=exp003"])
lost = hit[(hit["exp001"] == 1) & (hit["exp003"] == 0)].index
gold_docs = set(gold.loc[lost, "doc_id"])
with (ROOT / hcfg.paths.corpus).open(encoding="utf-8") as f:
    docs = [d for d in map(json.loads, f) if d["doc_id"] in gold_docs]
idx = ConditionIndex(docs, hcfg.retriever.conditions.year_ref_offsets)
pos = {d["doc_id"]: i for i, d in enumerate(docs)}
rows = []
for q in lost:
    c = conds[q]
    matched = idx.match_counts(c).get(pos[gold.loc[q, "doc_id"]], 0)
    rows.append({"qid": q, "조건 필드": ", ".join(sorted(c)), "조건 필드 수": len(c),
                 "정답 문서가 맞은 필드 수": matched,
                 "exp003 정답 순위": ranks["exp003"][q]})
pd.DataFrame(rows).set_index("qid")

,조건 필드,조건 필드 수,정답 문서가 맞은 필드 수,exp003 정답 순위
qid,,,,
047706-0007,session_number,1,0,6.0
048689-0018,meeting_name,1,0,9.0
049211-0016,committee_name,1,0,NaN
050951-0014,meeting_name,1,0,8.0
052269-0013,date,1,0,7.0
052392-0001,date,1,0,7.0


## 필드 2개 이상 일치(`min_fields: 2`)로 다시 실행

위 결과에서 밀린 6건은 모두 조건 필드가 하나였고 정답 문서는 그 조건에 맞지 않았다. 필드 하나로는 회의가 좁혀지지 않아 조건 목록이 그 조건에 맞는 문서만 남긴 dense 순위가 되고 dense 점수를 한 번 더 더하는 것으로 본다. 사용자 결정(2026-10-12)으로 필드가 2개 이상 맞은 문서만 조건 목록에 넣고, 같은 명령(`uv run python -m rag.search +exp=exp003`)으로 순위 파일을 다시 만들었다. 위 출력은 첫 설계(`min_fields` 없음)의 기록이다.

In [10]:
runs["exp003"] = read_run(RUNS["exp003"])
order["exp003"] = runs["exp003"].sort_values(["qid", "rank"]).groupby("qid")["doc_id"].apply(tuple)
same = order["exp001"] == order["exp003"]
pd.crosstab(has_cond, same, rownames=["조건 있음"], colnames=["순위 같음"])

순위 같음,True
조건 있음,
False,981
True,58


In [11]:
ranks["exp003"] = gold_ranks(runs["exp003"], gold, n_min)
scores["exp003"] = per_query(ranks["exp003"], ev.recall_ks, ev.mrr_k)
pd.DataFrame({
    "exp001 전체": scores["exp001"].mean(), "exp003 전체": scores["exp003"].mean(),
    "exp001 조건 있음": scores["exp001"].loc[cond_q].mean(),
    "exp003 조건 있음": scores["exp003"].loc[cond_q].mean(),
}).round(4)

,exp001 전체,exp003 전체,exp001 조건 있음,exp003 조건 있음
recall@1,0.9105,0.9105,0.9483,0.9483
recall@5,0.9808,0.9808,1.0000,1.0000
recall@10,0.9875,0.9875,1.0000,1.0000
mrr@10,0.9438,0.9438,0.9741,0.9741


In [12]:
n_fields = pd.Series({q: len(conds[q]) for q in cond_q})
print("조건 필드 수별 질의 수", n_fields.value_counts().sort_index().to_dict())
hit = pd.DataFrame({name: s["recall@5"] for name, s in scores.items()})
pd.crosstab(hit.loc[cond_q, "exp001"], hit.loc[cond_q, "exp003"],
            rownames=["exp001@5"], colnames=["exp003@5"])

조건 필드 수별 질의 수 {1: 55, 2: 3}


exp003@5,1.0
exp001@5,
1.0,58
